# Interactive Load Test — Decomposing the Locustfile

This notebook takes every piece of logic inside `locustfile.py` and makes it a separate, runnable cell.  
Nothing is automated. You decide when to move to the next step and can re-run any cell as many times as you want.

**Reading order of the locustfile:**
1. `SHORT_PROMPTS` / `LONG_PROMPTS` — the data sent in each request
2. `HEADERS` — the auth credential every request carries
3. `short_prompt` task — the most frequent request type (60%)
4. `long_prompt` task — the GPU-intensive one (20%)
5. `streaming_request` task — SSE stream (20%)
6. `wait_time = between(1, 3)` — think time between tasks
7. `--users N --spawn-rate M` — the concurrency knob

Each cell below corresponds to one of those pieces.

---
## Cell 1 — Imports and working directory

We import the same libraries Locust uses internally (`requests`, `random`, `time`) plus `threading` for the concurrency cells later.  
`os.chdir()` sets the kernel's working directory to the repo root so relative paths work.

In [1]:
import os, random, time, threading, json, sys
import requests

os.chdir("/workspace/vllm-gateway-native")
print("cwd:", os.getcwd())

cwd: /workspace/vllm-gateway-native


---
## Cell 2 — The prompt datasets

This is the first section of `locustfile.py`. Two lists of prompts:  
- `SHORT_PROMPTS` — one-line factual questions, expect short completions  
- `LONG_PROMPTS` — multi-paragraph technical topics, expect long completions  

Locust uses `random.choice()` on these to avoid sending the identical prompt repeatedly, which would give unrealistically fast results from vLLM's prefix cache.

In [2]:
SHORT_PROMPTS = [
    "What is the capital of France?",
    "Explain what a REST API is in one sentence.",
    "What does CPU stand for?",
    "What is the boiling point of water?",
    "Name three programming languages.",
]

LONG_PROMPTS = [
    "Write a detailed technical explanation of how transformer attention mechanisms work, covering scaled dot-product attention, multi-head attention, and the role of positional encodings.",
    "Explain the CAP theorem in distributed systems. Cover consistency, availability, and partition tolerance with real-world examples of databases that make different tradeoffs.",
    "Describe the complete lifecycle of an HTTP request from the moment a user types a URL in their browser to receiving the rendered page, including DNS resolution, TCP handshake, TLS, and HTTP/2 multiplexing.",
    "Explain how Kubernetes handles pod scheduling, including the role of the scheduler, node affinity, taints and tolerations, and resource requests vs limits.",
    "What are the tradeoffs between SQL and NoSQL databases? Cover consistency models, query patterns, scalability approaches, and give examples of when you would choose each.",
]

print(f"Short prompts: {len(SHORT_PROMPTS)}")
print(f"Long prompts:  {len(LONG_PROMPTS)}")
print(f"\nExample short: {random.choice(SHORT_PROMPTS)}")
print(f"Example long:  {random.choice(LONG_PROMPTS)[:80]}...")

Short prompts: 5
Long prompts:  5

Example short: What is the capital of France?
Example long:  Explain the CAP theorem in distributed systems. Cover consistency, availability,...


---
## Cell 3 — The auth header

Every request in the locustfile carries this header.  
The gateway's `auth.py` reads `X-API-Key` and checks it against the `API_KEYS` env var.  
A missing or wrong key returns HTTP 403 before the request even reaches vLLM.

In [3]:
GATEWAY = "http://localhost:8080"
HEADERS = {"X-API-Key": "key-dev-123", "Content-Type": "application/json"}

# Confirm the gateway accepts this key
r = requests.get(f"{GATEWAY}/health", headers={"X-API-Key": HEADERS["X-API-Key"]})
print(f"Gateway health: {r.status_code} {r.text}")

# Confirm a wrong key is rejected
r_bad = requests.post(f"{GATEWAY}/v1/chat/completions",
                      headers={"X-API-Key": "wrong-key", "Content-Type": "application/json"},
                      json={"model": "qwen2.5-3b", "messages": [{"role": "user", "content": "hi"}]})
print(f"Bad key response: {r_bad.status_code} {r_bad.text}")

Gateway health: 200 {"status":"ok"}
Bad key response: 403 {"detail":"Invalid API key"}


---
## Cell 4 — The `short_prompt` task (single request)

This is exactly the body of the `short_prompt` method in `StandardUser`, unwrapped from Locust.  
Weight 3 means Locust picks this task 60% of the time (3 out of 3+1+1).  
- `model`: must match `--served-model-name` in `run_vllm.sh`  
- `max_tokens: 128`: caps the output at 128 tokens — short factual answers  
- `stream: False`: wait for the full response before returning anything  

Run this cell multiple times. Each run picks a different prompt from `SHORT_PROMPTS`.

In [6]:
prompt = random.choice(SHORT_PROMPTS)

payload = {
    "model": "qwen2.5-3b",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 128,
    "stream": False
}

print(f"Prompt: {prompt}")
print("-" * 60)

t0 = time.time()
resp = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS)
latency = time.time() - t0

data = resp.json()
answer = data["choices"][0]["message"]["content"]
usage  = data.get("usage", {})

print(f"Answer: {answer}")
print("-" * 60)
print(f"Status:            {resp.status_code}")
print(f"Total latency:     {latency:.3f}s")
print(f"Prompt tokens:     {usage.get('prompt_tokens')}")
print(f"Completion tokens: {usage.get('completion_tokens')}")
print(f"Request-ID:        {resp.headers.get('X-Request-ID')}")

Prompt: What is the boiling point of water?
------------------------------------------------------------
Answer: The boiling point of water varies slightly depending on the pressure. At standard atmospheric pressure, which is 1 atmosphere (or 101.325 kilopascals), the boiling point of water is 100 degrees Celsius (212 degrees Fahrenheit). However, at higher or lower pressures, the boiling point can be higher or lower, respectively.
------------------------------------------------------------
Status:            200
Total latency:     1.633s
Prompt tokens:     37
Completion tokens: 76
Request-ID:        6676077a-4a7f-4308-9ad6-4aaa27cfc027


---
## Cell 5 — The `long_prompt` task (single request)

Same structure as `short_prompt` but with two key differences:  
- `max_tokens: 512` — allows a much longer completion  
- `timeout=120` — locustfile sets this explicitly because long responses can take up to 2 minutes under load  

This task has weight 1 (20% of traffic). It is the most GPU-memory-intensive request because it builds a large KV cache for both the long input prompt and the long output.  

Run this and notice the latency difference compared to Cell 4.

In [7]:
prompt = random.choice(LONG_PROMPTS)

payload = {
    "model": "qwen2.5-3b",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 512,
    "stream": False
}

print(f"Prompt ({len(prompt)} chars): {prompt[:120]}...")
print("-" * 60)

t0 = time.time()
resp = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS, timeout=120)
latency = time.time() - t0

data = resp.json()
answer = data["choices"][0]["message"]["content"]
usage  = data.get("usage", {})

print(f"Answer ({len(answer)} chars):\n{answer[:400]}..." if len(answer) > 400 else f"Answer:\n{answer}")
print("-" * 60)
print(f"Status:            {resp.status_code}")
print(f"Total latency:     {latency:.3f}s")
print(f"Prompt tokens:     {usage.get('prompt_tokens')}")
print(f"Completion tokens: {usage.get('completion_tokens')}")

Prompt (183 chars): Write a detailed technical explanation of how transformer attention mechanisms work, covering scaled dot-product attenti...
------------------------------------------------------------
Answer (2438 chars):
Transformer architectures, particularly the original Transformer model introduced by Vaswani et al., rely on attention mechanisms to compute their outputs. These mechanisms allow the model to weigh the importance of different elements of the input data, which is crucial for tasks like machine translation, natural language processing, and many others. Let's delve into the details of how transformer...
------------------------------------------------------------
Status:            200
Total latency:     10.948s
Prompt tokens:     59
Completion tokens: 512


---
## Cell 6 — The `streaming_request` task (single request, tokens printed as they arrive)

This is the SSE (Server-Sent Events) path. Two `stream=True` flags are at play:  
- In the JSON payload: tells vLLM to emit tokens one at a time instead of buffering the whole completion  
- In `requests.post(..., stream=True)`: tells the HTTP client not to buffer the response body  

Each line from the response body looks like `data: {"choices": [{"delta": {"content": "token"}}]}`.  
The final line is `data: [DONE]`.  

Watch the output cell: tokens should appear one by one as vLLM generates them.

In [8]:
prompt = random.choice(SHORT_PROMPTS)

payload = {
    "model": "qwen2.5-3b",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 256,
    "stream": True
}

print(f"Prompt: {prompt}")
print("-" * 60)
print("Streaming response (tokens arrive below as generated):")
print()

t0 = time.time()
ttft = None
token_count = 0

with requests.post(f"{GATEWAY}/v1/chat/completions",
                   json=payload, headers=HEADERS, stream=True) as resp:
    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode("utf-8") if isinstance(raw_line, bytes) else raw_line
        if not line.startswith("data: "):
            continue
        payload_str = line[len("data: "):]
        if payload_str.strip() == "[DONE]":
            break
        chunk = json.loads(payload_str)
        delta = chunk["choices"][0]["delta"]
        token = delta.get("content", "")
        if token:
            if ttft is None:
                ttft = time.time() - t0
            token_count += 1
            print(token, end="", flush=True)

total = time.time() - t0
print()
print("-" * 60)
print(f"Time to first token (TTFT): {ttft:.3f}s")
print(f"Total stream duration:       {total:.3f}s")
print(f"Tokens received:             {token_count}")
if token_count > 1 and total > ttft:
    print(f"Decode throughput:           {(token_count - 1) / (total - ttft):.1f} tok/s")

Prompt: What does CPU stand for?
------------------------------------------------------------
Streaming response (tokens arrive below as generated):

CPU stands for Central Processing Unit.
------------------------------------------------------------
Time to first token (TTFT): 0.054s
Total stream duration:       0.204s
Tokens received:             7
Decode throughput:           39.8 tok/s


---
## Cell 7 — The `wait_time = between(1, 3)` — one user looping through all three tasks

`wait_time = between(1, 3)` is the think-time Locust inserts between tasks for each user.  
This simulates a real user pausing before their next request — it controls how frequently a single user fires requests.

This cell runs one simulated user for **5 complete task iterations**: it picks a task using the same 3:1:1 weights Locust uses, executes it, waits 1–3 seconds, then picks the next task.  
You can see the timing between each request and what task was chosen each time.

In [11]:
# Reproduces Locust's weighted task selection: short=3, long=1, streaming=1
TASK_POOL = ["short"] * 3 + ["long"] * 1 + ["streaming"] * 1  # 5 entries, mirrors @task weights

NUM_ITERATIONS = 5   # Change this to run more loops
results = []

for i in range(NUM_ITERATIONS):
    task = random.choice(TASK_POOL)
    
    if task == "short":
        prompt = random.choice(SHORT_PROMPTS)
        payload = {"model": "qwen2.5-3b",
                   "messages": [{"role": "user", "content": prompt}],
                   "max_tokens": 128, "stream": False}
        t0 = time.time()
        r = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS)
        latency = time.time() - t0
        tokens = r.json().get("usage", {}).get("completion_tokens", "?")

    elif task == "long":
        prompt = random.choice(LONG_PROMPTS)
        payload = {"model": "qwen2.5-3b",
                   "messages": [{"role": "user", "content": prompt}],
                   "max_tokens": 512, "stream": False}
        t0 = time.time()
        r = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS, timeout=120)
        latency = time.time() - t0
        tokens = r.json().get("usage", {}).get("completion_tokens", "?")

    else:  # streaming
        prompt = random.choice(SHORT_PROMPTS)
        payload = {"model": "qwen2.5-3b",
                   "messages": [{"role": "user", "content": prompt}],
                   "max_tokens": 256, "stream": True}
        t0 = time.time()
        tokens = 0
        with requests.post(f"{GATEWAY}/v1/chat/completions",
                           json=payload, headers=HEADERS, stream=True) as r:
            for raw in r.iter_lines():
                line = raw.decode() if isinstance(raw, bytes) else raw
                if line.startswith("data: ") and line[6:] != "[DONE]":
                    d = json.loads(line[6:])["choices"][0]["delta"]
                    if d.get("content"):
                        tokens += 1
        latency = time.time() - t0

    results.append({"iter": i + 1, "task": task, "status": r.status_code,
                    "latency_s": round(latency, 3), "completion_tokens": tokens})
    print(f"[{i+1}/{NUM_ITERATIONS}] task={task:<10} status={r.status_code}  "
          f"latency={latency:.3f}s  tokens={tokens}")

    if i < NUM_ITERATIONS - 1:
        wait = random.uniform(1, 3)       # between(1, 3) from the locustfile
        print(f"           wait_time={wait:.2f}s ...")
        time.sleep(wait)

print("\nDone.")

[1/5] task=streaming  status=200  latency=2.786s  tokens=130
           wait_time=1.88s ...
[2/5] task=short      status=200  latency=0.735s  tokens=34
           wait_time=1.95s ...
[3/5] task=short      status=200  latency=2.731s  tokens=128
           wait_time=2.47s ...
[4/5] task=short      status=200  latency=0.184s  tokens=8
           wait_time=2.32s ...
[5/5] task=short      status=200  latency=0.184s  tokens=8

Done.


---
## Cell 8 — Simulate 2 concurrent users (--users 2)

In Locust, each user is an independent thread running its own task loop. Here we use Python's `threading` to do the same.  
Each thread picks tasks with the same 3:1:1 weights and waits 1–3 seconds between them — exactly as Locust would run them.

With 2 threads, two requests can be in-flight simultaneously. Watch the output: you'll see both threads logging concurrently.  
Look at the timestamps — they'll overlap, confirming real concurrency.

`DURATION` controls how long each thread runs.

In [12]:
NUM_USERS = 2
DURATION  = 45   # seconds

TASK_POOL = ["short"] * 3 + ["long"] + ["streaming"]
results_lock = threading.Lock()
all_results  = []

def run_user(user_id, duration):
    deadline = time.time() + duration
    while time.time() < deadline:
        task = random.choice(TASK_POOL)
        t0 = time.time()
        status = None
        tokens = "?"

        try:
            if task == "short":
                payload = {"model": "qwen2.5-3b",
                           "messages": [{"role": "user", "content": random.choice(SHORT_PROMPTS)}],
                           "max_tokens": 128, "stream": False}
                r = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS)
                status = r.status_code
                if status == 200:
                    tokens = r.json().get("usage", {}).get("completion_tokens", "?")

            elif task == "long":
                payload = {"model": "qwen2.5-3b",
                           "messages": [{"role": "user", "content": random.choice(LONG_PROMPTS)}],
                           "max_tokens": 512, "stream": False}
                r = requests.post(f"{GATEWAY}/v1/chat/completions", json=payload, headers=HEADERS, timeout=120)
                status = r.status_code
                if status == 200:
                    tokens = r.json().get("usage", {}).get("completion_tokens", "?")

            else:  # streaming
                payload = {"model": "qwen2.5-3b",
                           "messages": [{"role": "user", "content": random.choice(SHORT_PROMPTS)}],
                           "max_tokens": 256, "stream": True}
                tok = 0
                with requests.post(f"{GATEWAY}/v1/chat/completions",
                                   json=payload, headers=HEADERS, stream=True) as r:
                    status = r.status_code
                    for raw in r.iter_lines():
                        line = raw.decode() if isinstance(raw, bytes) else raw
                        if line.startswith("data: ") and line[6:] != "[DONE]":
                            d = json.loads(line[6:])["choices"][0]["delta"]
                            if d.get("content"):
                                tok += 1
                tokens = tok

        except Exception as e:
            status = f"ERR:{e}"

        latency = time.time() - t0
        row = {"user": user_id, "task": task, "status": status,
               "latency_s": round(latency, 3), "tokens": tokens,
               "ts": round(time.time(), 2)}

        with results_lock:
            all_results.append(row)
            print(f"user={user_id}  task={task:<10} status={status}  "
                  f"latency={latency:.3f}s  tokens={tokens}")
            sys.stdout.flush()

        time.sleep(random.uniform(1, 3))   # wait_time = between(1, 3)

all_results.clear()
print(f"Spawning {NUM_USERS} users for {DURATION}s ...\n")
threads = [threading.Thread(target=run_user, args=(i + 1, DURATION), daemon=True)
           for i in range(NUM_USERS)]
for t in threads: t.start()
for t in threads: t.join()

successes = [r for r in all_results if r["status"] == 200]
failures  = [r for r in all_results if r["status"] != 200]
latencies = [r["latency_s"] for r in successes]

print(f"\n--- {NUM_USERS} users, {DURATION}s summary ---")
print(f"Total requests:  {len(all_results)}")
print(f"Failures:        {len(failures)}")
if latencies:
    latencies.sort()
    n = len(latencies)
    print(f"Median latency:  {latencies[n // 2]:.3f}s")
    print(f"P95 latency:     {latencies[int(n * 0.95)]:.3f}s")
    print(f"Max latency:     {latencies[-1]:.3f}s")

Spawning 2 users for 45s ...

user=1  task=short      status=200  latency=0.203s  tokens=8
user=2  task=short      status=200  latency=2.738s  tokens=128
user=1  task=short      status=200  latency=0.183s  tokens=8
user=1  task=short      status=200  latency=0.712s  tokens=32
user=1  task=short      status=200  latency=0.711s  tokens=32
user=1  task=short      status=200  latency=0.608s  tokens=27
user=1  task=streaming  status=200  latency=0.199s  tokens=7
user=1  task=short      status=200  latency=0.188s  tokens=8
user=2  task=long       status=200  latency=11.114s  tokens=512
user=2  task=streaming  status=200  latency=4.314s  tokens=194
user=2  task=short      status=200  latency=0.599s  tokens=26
user=2  task=short      status=200  latency=0.206s  tokens=8
user=1  task=long       status=200  latency=11.203s  tokens=512
user=2  task=short      status=200  latency=2.621s  tokens=121
user=2  task=short      status=200  latency=0.800s  tokens=36
user=2  task=short      status=200  la

---
## Cell 9 — Scale to 5 concurrent users (--users 5 --spawn-rate 2)

Locust's `--spawn-rate 2` means it adds 2 new users per second until the target is reached.  
We replicate that here by staggering thread starts: a new thread launches every `1 / spawn_rate` seconds.

With 5 users in the same 3:1:1 mix, multiple requests will be in-flight simultaneously.  
The `long_prompt` requests from different users can overlap — those are the ones that create queue depth in vLLM's scheduler.

In [13]:
NUM_USERS  = 5
SPAWN_RATE = 2   # users per second — matches --spawn-rate 2 in run_benchmarks.sh
DURATION   = 90  # seconds

all_results.clear()
print(f"Spawning {NUM_USERS} users at {SPAWN_RATE}/s for {DURATION}s ...\n")

threads = []
for i in range(NUM_USERS):
    t = threading.Thread(target=run_user, args=(i + 1, DURATION), daemon=True)
    threads.append(t)
    t.start()
    if i < NUM_USERS - 1:
        time.sleep(1 / SPAWN_RATE)   # stagger launches to match spawn-rate

for t in threads: t.join()

successes = [r for r in all_results if r["status"] == 200]
failures  = [r for r in all_results if r["status"] != 200]
latencies = sorted([r["latency_s"] for r in successes])

print(f"\n--- {NUM_USERS} users, {DURATION}s summary ---")
print(f"Total requests:  {len(all_results)}")
print(f"Failures:        {len(failures)}")
if failures:
    print(f"Failure statuses: {set(r['status'] for r in failures)}")
if latencies:
    n = len(latencies)
    print(f"Median latency:  {latencies[n // 2]:.3f}s")
    print(f"P95 latency:     {latencies[int(n * 0.95)]:.3f}s")
    print(f"Max latency:     {latencies[-1]:.3f}s")

Spawning 5 users at 2/s for 90s ...

user=1  task=short      status=200  latency=0.185s  tokens=8
user=2  task=short      status=200  latency=0.182s  tokens=8
user=3  task=short      status=200  latency=0.191s  tokens=8
user=1  task=streaming  status=200  latency=4.446s  tokens=194
user=3  task=short      status=200  latency=2.925s  tokens=128
user=2  task=short      status=200  latency=2.915s  tokens=128
user=4  task=long       status=200  latency=11.661s  tokens=512
user=5  task=long       status=200  latency=11.633s  tokens=512
user=4  task=short      status=200  latency=0.832s  tokens=35
user=5  task=short      status=200  latency=0.872s  tokens=38
user=1  task=long       status=200  latency=11.671s  tokens=512
user=4  task=short      status=200  latency=0.193s  tokens=8
user=3  task=long       status=200  latency=11.679s  tokens=512
user=2  task=long       status=200  latency=11.656s  tokens=512
user=1  task=short      status=200  latency=0.193s  tokens=8
user=5  task=short      s

---
## Cell 10 — Scale to 10 concurrent users (--users 10 --spawn-rate 3)

The heaviest tier from `run_benchmarks.sh`. Spawn rate rises to 3 users/second — all 10 are active within ~4 seconds.  

At this concurrency, requests from all three task types will be in-flight simultaneously.  
The rate limiter in the gateway is set to 60 req/min per IP — with 10 users all sharing the same machine IP you may start seeing 429 responses.  
Any 429s will show up in the failures count at the end.

In [14]:
NUM_USERS  = 10
SPAWN_RATE = 3   # users per second — matches --spawn-rate 3
DURATION   = 90  # seconds

all_results.clear()
print(f"Spawning {NUM_USERS} users at {SPAWN_RATE}/s for {DURATION}s ...\n")

threads = []
for i in range(NUM_USERS):
    t = threading.Thread(target=run_user, args=(i + 1, DURATION), daemon=True)
    threads.append(t)
    t.start()
    if i < NUM_USERS - 1:
        time.sleep(1 / SPAWN_RATE)

for t in threads: t.join()

successes = [r for r in all_results if r["status"] == 200]
failures  = [r for r in all_results if r["status"] != 200]
latencies = sorted([r["latency_s"] for r in successes])

by_task = {}
for r in successes:
    by_task.setdefault(r["task"], []).append(r["latency_s"])

print(f"\n--- {NUM_USERS} users, {DURATION}s summary ---")
print(f"Total requests:  {len(all_results)}")
print(f"Failures:        {len(failures)}")
if failures:
    from collections import Counter
    print(f"Failure breakdown: {dict(Counter(str(r['status']) for r in failures))}")
if latencies:
    n = len(latencies)
    print(f"Overall median:  {latencies[n // 2]:.3f}s")
    print(f"Overall P95:     {latencies[int(n * 0.95)]:.3f}s")
    print(f"Overall max:     {latencies[-1]:.3f}s")
print()
for task, lats in sorted(by_task.items()):
    lats.sort()
    n = len(lats)
    print(f"  {task:<10}  n={n:<4}  median={lats[n//2]:.3f}s  "
          f"p95={lats[int(n*0.95)]:.3f}s  max={lats[-1]:.3f}s")

Spawning 10 users at 3/s for 90s ...

user=1  task=streaming  status=200  latency=0.185s  tokens=7
user=6  task=streaming  status=200  latency=0.209s  tokens=7
user=8  task=short      status=200  latency=1.003s  tokens=38
user=3  task=short      status=200  latency=3.155s  tokens=128
user=5  task=short      status=200  latency=3.163s  tokens=128
user=7  task=short      status=200  latency=3.120s  tokens=128
user=10  task=short      status=200  latency=2.970s  tokens=124
user=8  task=short      status=200  latency=0.211s  tokens=8
user=6  task=short      status=200  latency=3.085s  tokens=128
user=3  task=short      status=200  latency=0.774s  tokens=31
user=9  task=streaming  status=200  latency=4.451s  tokens=183
user=1  task=streaming  status=200  latency=5.468s  tokens=225
user=10  task=short      status=200  latency=0.473s  tokens=20
user=7  task=short      status=200  latency=3.148s  tokens=128
user=3  task=short      status=200  latency=0.211s  tokens=8
user=1  task=short      st

---
## Cell 11 — Three-tier comparison

After running Cells 8, 9, and 10, re-run this cell to see the numbers side by side.  
This cell reads `all_results` from the last run — to compare all three tiers you need to save results between runs.  

If you want to compare across runs, change the `NUM_USERS` / `DURATION` variables in each cell, run them in sequence, and paste the printed summary lines here manually — or extend the code below to accumulate results into a dict keyed by user count.

In [15]:
# Re-run this cell immediately after each load tier to capture a snapshot
# Then compare the printed blocks manually

successes = [r for r in all_results if r["status"] == 200]
failures  = [r for r in all_results if r["status"] != 200]
latencies = sorted(r["latency_s"] for r in successes)
by_task   = {}
for r in successes:
    by_task.setdefault(r["task"], []).append(r["latency_s"])

print(f"Requests: {len(all_results)}  |  Failures: {len(failures)}")
if latencies:
    n = len(latencies)
    print(f"Median: {latencies[n//2]:.3f}s  |  P95: {latencies[int(n*0.95)]:.3f}s  |  Max: {latencies[-1]:.3f}s")
print()
header = f"{'Task':<12} {'Count':>6} {'Median':>8} {'P95':>8} {'Max':>8}"
print(header)
print("-" * len(header))
for task, lats in sorted(by_task.items()):
    lats.sort()
    n = len(lats)
    print(f"{task:<12} {n:>6} {lats[n//2]:>8.3f} {lats[int(n*0.95)]:>8.3f} {lats[-1]:>8.3f}")

Requests: 278  |  Failures: 157
Median: 1.806s  |  P95: 12.450s  |  Max: 12.633s

Task          Count   Median      P95      Max
----------------------------------------------
long             19   12.384   12.633   12.633
short            74    0.900    3.163    3.215
streaming        28    0.900    5.125    5.468


---
## Cell 12 — Prometheus metrics snapshot post-load

Pull the gateway's `/metrics` endpoint to see the counters and histograms that Prometheus has been scraping.  
This shows the cumulative view of everything that happened since the gateway started — request counts, latency histogram buckets, token throughput, and rate-limited counts.

In [16]:
r = requests.get(f"{GATEWAY}/metrics")
gateway_lines = [l for l in r.text.splitlines()
                 if l.startswith("gateway_") or l.startswith("# HELP gateway_") or l.startswith("# TYPE gateway_")]
print("\n".join(gateway_lines))

# HELP gateway_requests_total Total requests through gateway
# TYPE gateway_requests_total counter
gateway_requests_total{api_key_prefix="key-dev-",endpoint="/v1/chat/completions",method="POST",status_code="200"} 242.0
# HELP gateway_requests_created Total requests through gateway
# TYPE gateway_requests_created gauge
gateway_requests_created{api_key_prefix="key-dev-",endpoint="/v1/chat/completions",method="POST",status_code="200"} 1.7792922199742465e+09
# HELP gateway_request_latency_seconds End-to-end latency at gateway layer
# TYPE gateway_request_latency_seconds histogram
gateway_request_latency_seconds_bucket{endpoint="/v1/chat/completions",le="0.1"} 0.0
gateway_request_latency_seconds_bucket{endpoint="/v1/chat/completions",le="0.25"} 69.0
gateway_request_latency_seconds_bucket{endpoint="/v1/chat/completions",le="0.5"} 72.0
gateway_request_latency_seconds_bucket{endpoint="/v1/chat/completions",le="1.0"} 113.0
gateway_request_latency_seconds_bucket{endpoint="/v1/chat/completions",l